### 10M Transformer: 3-Digit Addition in JAX
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eliasubz/jax-transformer/blob/main/math-transformer.ipynb)

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
from flax.training import train_state
import time
import random

print('JAX devices:', jax.devices())
print('JAX version:', jax.__version__)


Vocabulary & Tokenisation

In [ ]:
CHARS      = list('0123456789 +=')
VOCAB_SIZE = len(CHARS)          # 13


stoi = {c:i for i, c in enumerate(CHARS)}
itos = {i:c for i, c in enumerate(CHARS)}
PAD_TOK = "<PAD>"
itos[VOCAB_SIZE] = PAD_TOK
stoi[PAD_TOK] = VOCAB_SIZE
PAD_ID = VOCAB_SIZE

# 999 + 999 = 1998 (16 chars) + 4 to be able to make experiments with bigger numbers
SEQ_LEN = 20

# Methods
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[idx] for idx in ids if idx != VOCAB_SIZE)

# Sanity checks
for ch in CHARS: 
    i = stoi[ch]
    print(f"{ch} maps to {i} and {i} maps to {itos[i]}")

s = "100 + 200 = 300"
print('Sample encode/decode:', decode(encode(s)))

0 maps to 0 and 0 maps to 0
1 maps to 1 and 1 maps to 1
2 maps to 2 and 2 maps to 2
3 maps to 3 and 3 maps to 3
4 maps to 4 and 4 maps to 4
5 maps to 5 and 5 maps to 5
6 maps to 6 and 6 maps to 6
7 maps to 7 and 7 maps to 7
8 maps to 8 and 8 maps to 8
9 maps to 9 and 9 maps to 9
  maps to 10 and 10 maps to  
+ maps to 11 and 11 maps to +
= maps to 12 and 12 maps to =
Sample encode/decode: 100 + 200 = 300


Dataset


It will simply contain all string from "1 + 1 = 2" until "999 + 999 = 1899"

In [ ]:
import random
import numpy as np

X = []
for i in range(1000):
    for j in range(1000):
        X.append(f"{i} + {j} = {i+j}")


def make_example(x):

    # tokenize
    tokens = encode(x)                                  

    # Pad until SEQ_LEN
    padded = tokens + [PAD_ID] * (SEQ_LEN+1 - len(tokens)) 
    input  = np.array(padded[:-1])                         
    target = np.array(padded[1:])                          

    # Mask: 1 only on answer digits (after "=")
    equality_idx = tokens.index("=")
    mask = np.zeros(SEQ_LEN)                              
    mask[equality_idx+2:] = 1

    return input, target, mask


# sanity check
inp, trg, msk = make_example(X[1])
print(X[1], len(inp), len(trg), len(msk))


279 + 700 = 979 17 17 17


In [ ]:
# Build the Dataset
random.shuffle(X)
examples = [make_example(ex) for ex in X]
Input, Target, Mask = [ex for ex in zip(*examples)] 

split = int(len(X)*0.85)
inp_train = Input[:split]  
trg_train = Target[:split] 
msk_train = Mask[:split]   
inp_val   = Input[split:]   
trg_val   = Target[split:]  
msk_val   = Mask[split:]    

print(f'Train: {len(inp_train)}   Val: {len(inp_val)}')


Train: 850,000   Val: 150,000


Model: 10M transformer-decoder


<img src="causal_self-attention.png" width="200"/>

In [ ]:
D_MODEL     = 256
N_HEADS     = 8
N_LAYERS    = 6
D_FF        = 1024
TOTAL_VOCAB = VOCAB_SIZE + 1   # 14 (includes PAD)


class CausalSelfAttention(nn.Module):
    d_model: int
    n_heads: int

    @nn.compact
    def __call__(self, x):
        B, T, C = x.shape
        head_dim = C // self.n_heads

        qkv = nn.Dense(3*C, use_bias=False)(x)  # B, T, 3C
        q, k, v = jnp.split(qkv, 3, axis=-1)    # B, T, C

        q = q.reshape(B, T, self.n_heads, head_dim).transpose(0,2,1,3)
        k = k.reshape(B, T, self.n_heads, head_dim).transpose(0,2,1,3)
        v = v.reshape(B, T, self.n_heads, head_dim).transpose(0,2,1,3)  # B, h_d, T, head_dim

        qv = (q @ k.transpose(0,1,3,2)) * (head_dim ** -0.5)  # B, h_d, T, T
        causal = jnp.tril(jnp.ones((T, T), dtype=bool))       # T, T
        qv = jnp.where(causal[None, None], qv, -1e9)          # B, h_d, T, T
        qv = jax.nn.softmax(qv, axis=-1)                      # B, h_d, T, T

        attn = (qv @ v).transpose(0,2,1,3).reshape(B, T, C)  # B, T, C
        out  = nn.Dense(C, use_bias=False)(attn)
        return out


class TransformerBlock(nn.Module):
    # Uses pre-norm architecture
    d_model: int
    n_heads: int
    d_ff: int

    @nn.compact
    def __call__(self, x):
        x_csa = CausalSelfAttention(self.d_model, self.n_heads)(nn.LayerNorm()(x))
        x = x + x_csa

        x_ff = nn.Dense(self.d_ff)(nn.LayerNorm()(x))
        x_ff = nn.gelu(x_ff)
        x_ff = nn.Dense(self.d_model)(x_ff)
        return x + x_ff


class MathTransformer(nn.Module):
    d_model: int
    n_heads: int
    d_ff: int
    n_blocks: int
    vocab_size: int
    max_len: int

    @nn.compact
    def __call__(self, x):
        B, T = x.shape
        pos = nn.Embed(self.max_len,    self.d_model)(jnp.arange(T)[None, :])
        emb = nn.Embed(self.vocab_size, self.d_model)(x)
        h = emb + pos

        for _ in range(self.n_blocks):
            h = TransformerBlock(self.d_model, self.n_heads, self.d_ff)(h)

        # Because of prenorm we need to normalize after final block
        h   = nn.LayerNorm()(h)
        out = nn.Dense(self.vocab_size)(h)
        return out


model = MathTransformer(
    vocab_size=TOTAL_VOCAB, d_model=D_MODEL, n_heads=N_HEADS,
    n_blocks=N_LAYERS, d_ff=D_FF, max_len=SEQ_LEN,
)

key    = jax.random.PRNGKey(0)
dummy  = jnp.ones((1, SEQ_LEN - 1), dtype=jnp.int32)
params = model.init(key, dummy)["params"]
n_p    = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"Parameter count: {n_p:,}  (~{n_p/1e6:.1f} M)")


Optimizer

In [ ]:
BATCH_SIZE    = 512
EPOCHS        = 10
LEARNING_RATE = 3e-4
WARMUP_STEPS  = 1000

steps_per_epoch = len(inp_train) // BATCH_SIZE
total_steps     = steps_per_epoch * EPOCHS

# Linear warmup then cosine decay
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,  peak_value=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS, decay_steps=total_steps,
    end_value=LEARNING_RATE / 10,
)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(schedule, weight_decay=1e-2),
)
state = train_state.TrainState.create(
    apply_fn=model.apply, params=params, tx=optimizer,
)
print(f'Steps/epoch: {steps_per_epoch},  total steps: {total_steps}')

In [ ]:
def masked_loss (params, x, y, mask):
    logits = model.apply({'params': params}, x)
    ce = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    return (ce * mask).sum() / (mask.sum() + 1e-9)

@jax.jit
def train_step(state, x, y, mask):
    loss, grads = jax.value_and_grad(masked_loss) (state.params, x, y, mask)
    return state.apply_gradients(grads=grads), loss

def compute_acc(logits, y, mask):
    # complicated masked shit
    preds = logits.argmax(-1)
    token_ok = (preds == y) | (mask == 0)
    seq_ok     = token_ok.all(axis=-1)
    has_answer = (mask.sum(axis=-1) > 0)
    acc        = (seq_ok * has_answer).sum() / (has_answer.sum() + 1e-8)
    return acc


@jax.jit
def eval_step(params, x, y, mask):
    logits = model.apply({'params': params}, x)
    ce = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    loss = (ce * mask).sum() / (mask.sum() + 1e-9)
    return loss, compute_acc(logits, y, mask)

Training-Loop

In [ ]:
def train_and_evaluate(
        inp_train, trg_train, msk_train, inp_val, trg_val, msk_val, state, epochs
    ):

   
    for epoch in range(1, epochs + 1):
        best_eval_loss = 1e9

        perm = np.random.permuation(len(inp_train))
        tx, ty, tm = inp_train[perm], trg_train[perm], msk_train[perm]


        # Training
        acc_loss, nb = 0.0 , 0
        for i in range(0, len(tx)+BATCH_SIZE, BATCH_SIZE):

            xb = jnp.array(tx[i:i+BATCH_SIZE])
            yb = jnp.array(ty[i:i+BATCH_SIZE])
            mb = jnp.array(tm[i:i+BATCH_SIZE])

            state, loss = train_step(state, tx, ty, tm)
            acc_loss += loss
            nb += 1


        # Validation
        vl, va = [], []
        for i in range(0, len(inp_val)+BATCH_SIZE, BATCH_SIZE):
            
            metrics = eval_step(state.params, 
                      inp_val[i:i+BATCH_SIZE],
                      trg_val[i:i+BATCH_SIZE],
                      msk_val[i:i+BATCH_SIZE])
            vl.append(float(metrics[0]))
            va.append(float(metrics[1]))


        # Log Metrics to Weights & Biases
        print(f'Epoch {epoch:2d} | '
          f'train={acc_loss/nb:.4f} | '
          f'val={np.mean(vl):.4f} | '
          f'acc={np.mean(va)*100:.1f}')


    return state
inp_val = jnp.array(inp_val)
trg_val = jnp.array(trg_val)
msk_val = jnp.array(msk_val)

train_and_evaluate(inp_train, trg_train, msk_train, inp_val, trg_val, msk_val, state, EPOCHS)


Inference

In [ ]:
def generation(a, b, max_new=6):
    """Greedy decode the answer for 'a + b ='."""
    tokens = encode(f'{a} + {b} = ')
    for _ in range(max_new):
        inp    = tokens[-(SEQ_LEN - 1):]
        padded = inp + [PAD_ID] * max(0, SEQ_LEN  - len(inp))
        x      = jnp.array(padded)[None, :]
        logits = model.apply({'params': state.params}, x, deterministic=True)
        nxt    = int(logits[0, len(inp) - 1].argmax())
        if nxt == PAD_ID:
            break
        tokens.append(nxt)
    return decode(tokens)

for i in range(6): 
    a = random.rand(1000)
    b = random.rand(1000)
    res = generation(a, b)
    print(f'{a} + {b} = {a+b}')
    print(res)

Testset

In [ ]:
test = inp_val[:2000]
